# **Exploración de los datos**
---

## **1. Librerías**
---

In [2]:
import pandas as pd
import os
import sys
import argparse 
from datetime import datetime
from dateutil import parser
import matplotlib.pyplot as plt
import numpy as np

## **2. Análisis de la base de datos**
---

### **2.1 Cargue de la base de datos**
---

In [4]:
ruta_data_original = 'C:\\Users\\cecor\\OneDrive - Universidad Nacional de Colombia\\GitHub\\MSc-Thesis-Financial-Fraud-Detection-Models\\data\\original'
archivo_patterns = 'HI-Large_patterns.txt'
ruta_archivo = os.path.join(ruta_data_original, archivo_patterns)

# 2. Lista para guardar los datos procesados
data_procesada = []
patron_actual = "Desconocido"

# 3. Nombres de columnas
columnas = [
    'timestamp', 'from_bank', 'from_account', 'to_bank', 'to_account', 
    'amount_received', 'receiving_currency', 'amount_paid', 
    'payment_currency', 'payment_format', 'is_laundering'
]

print("Leyendo y procesando el archivo de patrones...")

if os.path.exists(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        for linea in f:
            linea = linea.strip()
            
            # Detectar inicio de un nuevo patrón
            if linea.startswith("BEGIN LAUNDERING ATTEMPT"):
                # Extraer el nombre del patrón (ej: "STACK", "CYCLE")
                # La línea es: "BEGIN LAUNDERING ATTEMPT - STACK"
                try:
                    patron_actual = linea.split('-')[1].strip()
                except:
                    patron_actual = linea.replace("BEGIN LAUNDERING ATTEMPT -", "").strip()
            
            # Detectar fin del patrón
            elif linea.startswith("END LAUNDERING ATTEMPT"):
                patron_actual = "Desconocido"
            
            # Si es una línea de datos (empieza con fecha/año 2022)
            elif linea.startswith("2022"):
                valores = linea.split(',')
                
                # Agregamos el "patron_actual" como una columna nueva al final
                fila = valores + [patron_actual]
                data_procesada.append(fila)

    # 4. Crear el DataFrame final
    # Agregamos 'laundering_pattern' a la lista de columnas
    columnas_finales = columnas + ['laundering_pattern']
    df_patterns = pd.DataFrame(data_procesada, columns=columnas_finales)

    print(f"¡Proceso terminado! Dimensiones: {df_patterns.shape}")
    print("\nEjemplo de los datos enriquecidos:")
    
    # Mostramos solo las columnas clave para verificar
    try:
        display(df_patterns[['timestamp', 'from_bank', 'to_bank', 'laundering_pattern', 'is_laundering']].head(10))
    except:
        print(df_patterns[['timestamp', 'from_bank', 'to_bank', 'laundering_pattern', 'is_laundering']].head(10))

else:
    print("No se encuentra el archivo.")

Leyendo y procesando el archivo de patrones...
¡Proceso terminado! Dimensiones: (137782, 12)

Ejemplo de los datos enriquecidos:


,timestamp,from_bank,to_bank,laundering_pattern,is_laundering
0,2022/08/09 05:14,00952,0111632,STACK,1
1,2022/08/13 13:09,0111632,008456,STACK,1
2,2022/08/15 07:40,0118693,013729,STACK,1
3,2022/08/15 14:19,013729,0123621,STACK,1
4,2022/08/13 12:40,0024750,0213834,STACK,1
5,2022/08/22 06:34,0213834,000,STACK,1
6,2022/08/01 00:19,0134266,0036925,CYCLE: Max 12 hops,1
7,2022/08/01 13:05,0036925,0119211,CYCLE: Max 12 hops,1
8,2022/08/03 13:28,0119211,0132965,CYCLE: Max 12 hops,1
9,2022/08/09 02:32,0132965,0137089,CYCLE: Max 12 hops,1


### **2.2 Análisis de la base de datos**
---

In [3]:
# Cambio el nombre de las columnas
df_accounts.columns=df_accounts.columns.str.strip().str.lower().str.replace(' ', '_')
df_accounts.columns

Index(['bank_name', 'bank_id', 'account_number', 'entity_id', 'entity_name'], dtype='str')

In [4]:
# Verificar valores nulos
for i in df_accounts.columns:
    print(f"El número de valores nulos para la columna: {i}")
    print(f" es de: {df_accounts[i].isnull().sum()}")

El número de valores nulos para la columna: bank_name
 es de: 0
El número de valores nulos para la columna: bank_id
 es de: 0
El número de valores nulos para la columna: account_number
 es de: 0
El número de valores nulos para la columna: entity_id
 es de: 0
El número de valores nulos para la columna: entity_name
 es de: 0


In [5]:
# Mirar la distribución de las variables
for i in df_accounts.columns:
    if df_accounts[i].dtype == 'object':
        print(f"Columna: {i}")
        print(df_accounts[i].nunique())

In [6]:
print(f"Dado que el número de filas del conjunto de datos es de:{df_accounts.shape[0]:,}.") 
print(f"y el número de cuentas únicas es de: {df_accounts['account_number'].nunique():,}.  Se concluye que hay: {df_accounts.shape[0] - df_accounts['account_number'].nunique():,} cuentas repetidas.")

Dado que el número de filas del conjunto de datos es de:2,126,855.
y el número de cuentas únicas es de: 2,110,228.  Se concluye que hay: 16,627 cuentas repetidas.


In [7]:
# Validación de duplicados en account_key
duplicados_cuenta=df_accounts['account_number'].value_counts()
print(duplicados_cuenta[duplicados_cuenta>1].head(2))

account_number
82EE0ED30    4
8322DBCF0    4
Name: count, dtype: int64


In [8]:
# Creo una llave unica para cada cuenta combinando bank_id y account_number
# No hay valores duplicados después de crear la llave única, lo que es una buena señal.
df_accounts["account_key"]=df_accounts["bank_id"].astype(str)+"-"+df_accounts["account_number"].astype(str)
duplicados_cuenta_key=df_accounts['account_key'].value_counts()
if duplicados_cuenta_key[duplicados_cuenta_key>1].empty:
    print("No hay cuentas duplicadas después de crear la llave única.")
else:
    print("Hay cuentas duplicadas después de crear la llave única:")
    print(duplicados_cuenta_key[duplicados_cuenta_key>1])

No hay cuentas duplicadas después de crear la llave única.


In [9]:
# tipo de corporacion
df_accounts['entity_type'] = df_accounts['entity_name'].str.split('#').str[0].str.strip()
df_accounts["entity_type"].value_counts()

entity_type
Partnership            745341
Corporation            692680
Sole Proprietorship    680013
Country                  5689
Individual               3104
Direct                     28
Name: count, dtype: int64

In [10]:
# Las entidades pueden tener más de 1 banco adscrito
cuentas_por_entidad = df_accounts.groupby('entity_id')['bank_id'].nunique().reset_index()
cuentas_por_entidad.columns = ['entity_id', 'num_bancos_distintos']
cuentas_por_entidad.head(3)

,entity_id,num_bancos_distintos
0,2AA02E7E570,2
1,2AA02E7E5F0,15
2,2AA02E7E650,2


In [11]:
# Unir esto al DataFrame original
df_accounts = df_accounts.merge(cuentas_por_entidad, on='entity_id', how='left')

In [12]:
pd.concat([df_accounts.head(2), df_accounts.tail(2)])
df_accounts.head()

,bank_name,bank_id,account_number,entity_id,entity_name,account_key,entity_type,num_bancos_distintos
0,Portugal Bank #500,240522,82655C500,2AA04EEC5D0,Corporation #82502,240522-82655C500,Corporation,10
1,First Bank of Laramie,339367,8505ED380,2AA06B8AC80,Partnership #15630,339367-8505ED380,Partnership,21
2,National Bank of Helena,368763,826283D80,2AA066499F0,Corporation #12918,368763-826283D80,Corporation,18
3,Switzerland Bank #2454,3174937,842090C80,2AA06712690,Corporation #135403,3174937-842090C80,Corporation,477
4,China Bank #561,53267,817D00980,2AA053417D0,Corporation #103595,53267-817D00980,Corporation,15


### **2.3 Exportar la base de datos**
---

In [13]:
ruta_data_limpio = 'C:\\Users\\cecor\\OneDrive - Universidad Nacional de Colombia\\GitHub\\MSc-Thesis-Financial-Fraud-Detection-Models\\data\\processed'
archivo_cuentas = 'HI-Large_accounts_limpio'
ruta_archivo = os.path.join(ruta_data_limpio, archivo_cuentas + '.parquet')
#
try:
    # index=False para no columna extra
    df_accounts.to_parquet(ruta_archivo, index=False, engine='pyarrow')
    print(f"¡Archivo exportado exitosamente en:\n{ruta_archivo}")    
except Exception as e:
    print(f"Ocurrió un error al exportar: {e}")

¡Archivo exportado exitosamente en:
C:\Users\cecor\OneDrive - Universidad Nacional de Colombia\GitHub\MSc-Thesis-Financial-Fraud-Detection-Models\data\processed\HI-Large_accounts_limpio.parquet
